# 06 — Three-seed comparison (run_02)

Reruns the main generators with seeds 42, 43, 44 and reports mean ± SD, so the comparison is not a single lucky run.

Generators: SMOTE-NC, Gaussian copula, CTGAN, ExGAN-noshift, ExGAN-MA, ExGAN-MA-joint (final method).
ExGAN-MA and ExGAN-MA-joint come from the same trained model (only the mode sampling differs).

**Resumable:** every (generator, seed) is saved to `synthetic/seeds/` as soon as it finishes. If Colab disconnects,
run all cells again and finished ones are skipped.

Runtime: GPU (T4). Roughly 1 to 1.5 hours in total.

In [ ]:
# Cell 1 — install, paths and settings
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install sdv imbalanced-learn
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")      # local test only

OUT = f"{BASE}/genai"
SEED_DIR = f"{OUT}/synthetic/seeds"
RES_DIR = f"{OUT}/results"

SEEDS = (42, 43, 44)
GENERATORS = ["SMOTE-NC", "GaussianCopula", "CTGAN", "ExGAN-noshift", "ExGAN-MA"]   # ExGAN-MA also writes ExGAN-MA-joint
N_SYN = 1000

# same settings as 02 / 03 / 04 on run_02
CTGAN_EPOCHS = 300
MA_EPOCHS = 150                  # 04
NOSHIFT_EPOCHS = 45 + 10 * 15    # 03: INIT_EPOCHS + K * SHIFT_EPOCHS (K = 10 on run_02)
BATCH, Z_DIM, HIDDEN = 128, 64, 256
N_CRITIC, GP_W, LR = 5, 10.0, 2e-4
GUMBEL_TAU, LAMBDA_MODE = 0.2, 1.0

if os.environ.get("SMOKE"):      # tiny local test
    CTGAN_EPOCHS, MA_EPOCHS, NOSHIFT_EPOCHS, N_SYN = 2, 2, 2, 200

In [ ]:
# Cell 2 — load data, shared helpers (mode flags, codec, EVT)
import json, time, math, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import genpareto
warnings.filterwarnings("ignore")
os.makedirs(SEED_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

meta  = json.load(open(f"{OUT}/meta.json"))
train = pd.read_parquet(f"{OUT}/train.parquet")
val   = pd.read_parquet(f"{OUT}/val.parquet")
THR, MODES = meta["plos_threshold"], meta["modes"]
CAT, BIN = meta["cat_cols"], meta["bin_cols"]
NUM = meta["num_cols"] + ["target"]
GEN_COLS = NUM + CAT + BIN
MODE_NAMES = list(MODES.keys()); NM = len(MODE_NAMES)
MODE_FLAGS = sorted({c for v in MODES.values() for c in v if c in BIN})
NONMODE = [c for c in BIN if c not in MODE_FLAGS]
DEV = "cuda" if torch.cuda.is_available() else "cpu"
real_tail = train[train["is_plos"] == 1][GEN_COLS].reset_index(drop=True)

def mode_matrix(d):
    return np.stack([d[[c for c in MODES[k] if c in d.columns]].max(axis=1).values for k in MODE_NAMES], 1).astype(np.float32)

def mode_flags(d):
    f = pd.DataFrame(mode_matrix(d), columns=MODE_NAMES, index=d.index).astype(int)
    f["unexplained"] = (f.sum(axis=1) == 0).astype(int)
    return f

def clean(df):
    df = df[GEN_COLS].copy()
    for c in BIN: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().clip(0, 1).astype(int)
    for c in CAT: df[c] = df[c].astype(str)
    df["anchor_age"] = pd.to_numeric(df["anchor_age"], errors="coerce").round()
    df["target"] = pd.to_numeric(df["target"], errors="coerce").round()
    return df.dropna(subset=["anchor_age", "target"]).reset_index(drop=True)

def keep_valid(df, n):
    valid = df["target"] > THR
    return df[valid].head(n).reset_index(drop=True)

class Codec:
    def __init__(self, df):
        self.age_mu, self.age_sd = df["anchor_age"].mean(), df["anchor_age"].std()
        self.age_min, self.age_max = df["anchor_age"].min(), df["anchor_age"].max()
        logl = np.log(df["target"].clip(lower=1))
        self.l_mu, self.l_sd = logl.mean(), logl.std()
        self.levels = {c: sorted(df[c].astype(str).unique()) for c in CAT}
        self.cat_sizes = [len(self.levels[c]) for c in CAT]
        self.n_bin = len(BIN)
        self.dim = 1 + sum(self.cat_sizes) + 2 * self.n_bin
    def encode_x(self, df):
        parts = [((df["anchor_age"].values - self.age_mu) / self.age_sd)[:, None]]
        for c in CAT:
            v = df[c].astype(str).values
            parts.append(np.stack([(v == l) for l in self.levels[c]], 1).astype(np.float32))
        b = df[BIN].values.astype(np.float32)
        parts.append(np.stack([1 - b, b], 2).reshape(len(df), -1))
        return np.concatenate(parts, 1).astype(np.float32)
    def encode_e(self, los):
        return ((np.log(np.clip(los, 1, None)) - self.l_mu) / self.l_sd).astype(np.float32)[:, None]
    def decode(self, x, los):
        out, i = {}, 1
        out["anchor_age"] = np.clip(np.round(x[:, 0] * self.age_sd + self.age_mu), self.age_min, self.age_max)
        out["target"] = np.round(los)
        for c, s in zip(CAT, self.cat_sizes):
            out[c] = np.array(self.levels[c])[x[:, i:i + s].argmax(1)]; i += s
        flags = x[:, i:].reshape(len(x), self.n_bin, 2).argmax(2)
        df = pd.concat([pd.DataFrame(out), pd.DataFrame(flags, columns=BIN)], axis=1)
        return df[GEN_COLS]

codec = Codec(train)

# EVT step: GPD on real PLOS LOS above the threshold (fixed data, so identical for every seed)
xi, _, sigma = genpareto.fit(real_tail["target"].values - THR, floc=0)
LOS_CAP = 1.2 * real_tail["target"].max()
def sample_los_gpd(n, rng):
    los = THR + genpareto.rvs(xi, loc=0, scale=sigma, size=n, random_state=rng)
    return np.clip(np.maximum(np.round(los), math.floor(THR) + 1), None, LOS_CAP)

M_tail = mode_matrix(real_tail)
def sample_modes_given_los(los, rng, n_bins=10):
    real_los = real_tail["target"].values
    edges = np.unique(np.quantile(real_los, np.linspace(0, 1, n_bins + 1)))
    real_bin = np.clip(np.searchsorted(edges, real_los, side="right") - 1, 0, len(edges) - 2)
    req_bin = np.clip(np.searchsorted(edges, los, side="right") - 1, 0, len(edges) - 2)
    out = np.empty((len(los), NM), np.float32)
    for b in np.unique(req_bin):
        pool = np.where(real_bin == b)[0]
        if len(pool) == 0: pool = np.arange(len(real_los))
        out[req_bin == b] = M_tail[rng.choice(pool, size=(req_bin == b).sum())]
    return out

print(f"Device {DEV} | train {len(train)} | PLOS {len(real_tail)} | threshold {THR:.2f} d | GPD xi {xi:.3f}, sigma {sigma:.2f}")

In [ ]:
# Cell 3 — GAN networks (one class, with or without mode conditioning)
bin_start = 1 + sum(codec.cat_sizes)
mode_bin_idx = [[BIN.index(c) for c in MODES[k] if c in BIN] for k in MODE_NAMES]

class Generator(nn.Module):
    def __init__(self, n_cond):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(Z_DIM + n_cond, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, codec.dim))
    def forward(self, z, c):
        h = self.net(torch.cat([z, c], 1))
        outs, i = [h[:, :1]], 1
        for s in codec.cat_sizes:
            outs.append(F.gumbel_softmax(h[:, i:i + s], tau=GUMBEL_TAU)); i += s
        b = h[:, i:].reshape(-1, codec.n_bin, 2)
        outs.append(F.gumbel_softmax(b, tau=GUMBEL_TAU, dim=2).reshape(len(h), -1))
        return torch.cat(outs, 1)

class Critic(nn.Module):
    def __init__(self, n_cond):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(codec.dim + n_cond, HIDDEN), nn.LeakyReLU(0.2),
                                 nn.Linear(HIDDEN, HIDDEN), nn.LeakyReLU(0.2), nn.Linear(HIDDEN, 1))
    def forward(self, x, c):
        return self.net(torch.cat([x, c], 1))

def grad_penalty(D, real, fake, c):
    a = torch.rand(len(real), 1, device=DEV)
    mix = (a * real + (1 - a) * fake).requires_grad_(True)
    g = torch.autograd.grad(D(mix, c).sum(), mix, create_graph=True)[0]
    return ((g.norm(2, dim=1) - 1) ** 2).mean()

def mode_probs(x):
    p1 = x[:, bin_start + 1::2]
    return torch.stack([p1[:, idx].max(1).values for idx in mode_bin_idx], 1)

def new_models(n_cond):
    G, D = Generator(n_cond).to(DEV), Critic(n_cond).to(DEV)
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.9))
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.9))
    return G, D, optG, optD

def train_gan(G, D, X, C, epochs, optG, optD, sample_w=None, M=None):
    """WGAN-GP. sample_w=None: shuffled epochs (03). sample_w given: training-by-sampling + mode loss (04)."""
    X, C = torch.tensor(X, device=DEV), torch.tensor(C, device=DEV)
    if M is not None: M = torch.tensor(M, device=DEV)
    n = len(X); steps = math.ceil(n / BATCH)
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEV)
        for b in range(steps):
            idx = (torch.multinomial(sample_w, BATCH, replacement=True).to(DEV)
                   if sample_w is not None else perm[b * BATCH:(b + 1) * BATCH])
            real, c, nb = X[idx], C[idx], len(idx)
            for _ in range(N_CRITIC):
                fake = G(torch.randn(nb, Z_DIM, device=DEV), c).detach()
                lossD = D(fake, c).mean() - D(real, c).mean() + GP_W * grad_penalty(D, real, fake, c)
                optD.zero_grad(); lossD.backward(); optD.step()
            fake = G(torch.randn(nb, Z_DIM, device=DEV), c)
            lossG = -D(fake, c).mean()
            if M is not None:
                lossG = lossG + LAMBDA_MODE * F.binary_cross_entropy(mode_probs(fake).clamp(1e-6, 1 - 1e-6), M[idx])
            optG.zero_grad(); lossG.backward(); optG.step()
        if (ep + 1) % 50 == 0 or ep + 1 == epochs:
            print(f"      epoch {ep + 1}/{epochs} | lossD {lossD.item():.3f}", flush=True)

@torch.no_grad()
def generate(G, C):
    G.eval(); x = G(torch.randn(len(C), Z_DIM, device=DEV), torch.tensor(C, device=DEV)).cpu().numpy(); G.train()
    return x

X_all = codec.encode_x(train[GEN_COLS])
E_all = codec.encode_e(train["target"].values.astype(float))
M_train = mode_matrix(train)
pat = pd.Series(["".join(map(str, r.astype(int))) for r in M_train])
freq = pat.map(pat.value_counts()).values.astype(float)
row_w = np.log(freq + 1) / freq
row_w = torch.tensor(row_w / row_w.sum(), dtype=torch.float)

In [ ]:
# Cell 4 — one function per generator; each returns {variant: (rows, extra_info)}
def seed_all(seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    return np.random.default_rng(seed)

def run_smote(seed):
    from imblearn.over_sampling import SMOTENC
    seed_all(seed)
    X = train[GEN_COLS].copy()
    for c in BIN: X[c] = X[c].astype(str)
    y = train["is_plos"].values
    sm = SMOTENC(categorical_features=[X.columns.get_loc(c) for c in CAT + BIN],
                 sampling_strategy={1: int(y.sum()) + N_SYN}, k_neighbors=5, random_state=seed)
    X_res, _ = sm.fit_resample(X, y)
    return {"SMOTE-NC": (keep_valid(clean(X_res.iloc[len(X):]), N_SYN), {})}

def run_sdv(name, seed):
    from sdv.metadata import Metadata
    from sdv.single_table import GaussianCopulaSynthesizer, CTGANSynthesizer
    seed_all(seed)
    tt = real_tail.copy()
    for c in BIN: tt[c] = tt[c].astype(str)
    md_ = Metadata.detect_from_dataframe(tt, table_name="plos")
    for c in CAT + BIN: md_.update_column(column_name=c, sdtype="categorical", table_name="plos")
    for c in NUM: md_.update_column(column_name=c, sdtype="numerical", table_name="plos")
    if name == "GaussianCopula":
        m = GaussianCopulaSynthesizer(md_, enforce_min_max_values=True)
    else:
        m = CTGANSynthesizer(md_, epochs=CTGAN_EPOCHS, batch_size=100, enforce_min_max_values=True,
                             verbose=False, cuda=torch.cuda.is_available())
    m.fit(tt)
    m._set_random_state(seed)   # SDV otherwise samples with a fixed internal seed, so every seed would look identical
    return {name: (keep_valid(clean(m.sample(num_rows=3 * N_SYN)), N_SYN), {})}

def run_noshift(seed):
    rng = seed_all(seed)
    G, D, optG, optD = new_models(1)
    train_gan(G, D, X_all, E_all, NOSHIFT_EPOCHS, optG, optD)
    e = sample_los_gpd(N_SYN, rng)
    return {"ExGAN-noshift": (codec.decode(generate(G, codec.encode_e(e)), e), {})}

def run_ma(seed):
    rng = seed_all(seed)
    G, D, optG, optD = new_models(1 + NM)
    C_all = np.concatenate([E_all, M_train], 1)
    train_gan(G, D, X_all, C_all, MA_EPOCHS, optG, optD, sample_w=row_w, M=M_train)
    e = sample_los_gpd(N_SYN, rng)
    out = {}
    for variant, m_req in [("ExGAN-MA", M_tail[rng.integers(0, len(M_tail), N_SYN)]),
                           ("ExGAN-MA-joint", sample_modes_given_los(e, rng))]:
        syn = codec.decode(generate(G, np.concatenate([codec.encode_e(e), m_req], 1)), e)
        out[variant] = (syn, {"mode_consistency": float((mode_matrix(syn) == m_req).all(1).mean())})
    return out

RUNNERS = {"SMOTE-NC": run_smote, "GaussianCopula": lambda s: run_sdv("GaussianCopula", s),
           "CTGAN": lambda s: run_sdv("CTGAN", s), "ExGAN-noshift": run_noshift, "ExGAN-MA": run_ma}
VARIANTS = {"ExGAN-MA": ["ExGAN-MA", "ExGAN-MA-joint"]}

In [ ]:
# Cell 5 — run everything (resumable: finished (generator, seed) pairs are skipped)
LOG = f"{SEED_DIR}/_log.json"
log = json.load(open(LOG)) if os.path.exists(LOG) else {}
path = lambda v, s: f"{SEED_DIR}/{v}_s{s}.parquet"

t_all = time.time()
for seed in SEEDS:
    for gen in GENERATORS:
        variants = VARIANTS.get(gen, [gen])
        if all(os.path.exists(path(v, seed)) for v in variants):
            print(f"skip  {gen:15s} seed {seed} (already saved)"); continue
        print(f"run   {gen:15s} seed {seed} ...", flush=True)
        t0 = time.time()
        res = RUNNERS[gen](seed)
        t = time.time() - t0
        for v, (syn, info) in res.items():
            syn.to_parquet(path(v, seed), index=False)
            log[f"{v}_s{seed}"] = {"fit_s": t, **info}
        json.dump(log, open(LOG, "w"), indent=1)    # save after every run, survives disconnects
        print(f"      done in {t / 60:.1f} min | rows {[len(r[0]) for r in res.values()]}", flush=True)
print(f"All runs finished. This session: {(time.time() - t_all) / 60:.1f} min")

In [ ]:
# Cell 6 — metrics for every (generator, seed), same definitions as 04 cell 7
from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors

real_flags = mode_flags(real_tail).mean()
EXT_CUT = real_tail["target"].quantile(0.90)
real_ext_flags = mode_flags(real_tail[real_tail["target"] >= EXT_CUT]).mean()
age_mu, age_sd = real_tail["anchor_age"].mean(), real_tail["anchor_age"].std()
los_mu, los_sd = real_tail["target"].mean(), real_tail["target"].std()
cat_levels = {c: sorted(real_tail[c].astype(str).unique()) for c in CAT}
def encode(d):
    parts = [((d["anchor_age"] - age_mu) / age_sd).to_frame(), ((d["target"] - los_mu) / los_sd).to_frame(), d[BIN].astype(float)]
    for c in CAT:
        parts.append(pd.DataFrame({f"{c}={l}": (d[c].astype(str) == l).astype(float) for l in cat_levels[c]}, index=d.index))
    return pd.concat(parts, axis=1).values
nn_ = NearestNeighbors(n_neighbors=1).fit(encode(real_tail))
holdout = val[val["is_plos"] == 1][GEN_COLS].reset_index(drop=True)
dcr_holdout = np.median(nn_.kneighbors(encode(holdout))[0])

def corr_err(a, b, cols):
    d = np.abs(a[cols].corr().fillna(0).values - b[cols].corr().fillna(0).values)
    return d[np.triu_indices_from(d, 1)].mean()
real_mf = mode_flags(real_tail)
def within_mode_err(s):
    sf, errs = mode_flags(s), []
    for k in MODE_NAMES:
        r, g = real_tail[real_mf[k] == 1], s[sf[k] == 1]
        if len(r) >= 20 and len(g) >= 20:
            errs.append((r[NONMODE].mean() - g[NONMODE].mean()).abs().mean())
    return np.mean(errs) if errs else np.nan

def metrics(s):
    s = s[GEN_COLS].copy()
    for c in BIN: s[c] = s[c].astype(int)
    f = mode_flags(s).mean(); ext = s["target"] >= EXT_CUT
    d_syn = nn_.kneighbors(encode(s))[0].ravel()
    return {"n_rows": len(s),
        "LOS_KS": ks_2samp(real_tail["target"], s["target"]).statistic,
        "extreme_cov": (s["target"] > EXT_CUT).mean(), "LOS_max": s["target"].max(),
        **{f"mode_{k}": f[k] for k in f.index},
        "mode_abs_err": (f - real_flags).abs().mean(),
        "extreme_mode_err": (mode_flags(s[ext]).mean() - real_ext_flags).abs().mean() if ext.sum() >= 20 else np.nan,
        "nonmode_prev_err": (s[NONMODE].mean() - real_tail[NONMODE].mean()).abs().mean(),
        "nonmode_corr_err": corr_err(s, real_tail, ["anchor_age", "target"] + NONMODE),
        "within_mode_err": within_mode_err(s),
        "exact_copies": (d_syn < 1e-6).mean(), "DCR_ratio": np.median(d_syn) / dcr_holdout}

ORDER = ["SMOTE-NC", "GaussianCopula", "CTGAN", "ExGAN-noshift", "ExGAN-MA", "ExGAN-MA-joint"]
rows = []
for v in ORDER:
    for seed in SEEDS:
        if not os.path.exists(path(v, seed)): print(f"missing {v} seed {seed}"); continue
        info = log.get(f"{v}_s{seed}", {})
        rows.append({"generator": v, "seed": seed, **metrics(pd.read_parquet(path(v, seed))),
                     "mode_consistency": info.get("mode_consistency", np.nan), "fit_s": info.get("fit_s", np.nan)})
raw = pd.DataFrame(rows)
raw.to_csv(f"{RES_DIR}/06_seeds_raw.csv", index=False)
print(f"Real PLOS {len(real_tail)} | extreme cut {EXT_CUT:.0f} d | holdout DCR {dcr_holdout:.2f} | saved 06_seeds_raw.csv")

In [ ]:
# Cell 7 — mean ± SD over seeds, and the scorecard per seed
KEY = ["LOS_KS", "extreme_cov", "LOS_max", "mode_abs_err", "extreme_mode_err", "nonmode_prev_err",
       "nonmode_corr_err", "within_mode_err", "DCR_ratio", "exact_copies", "mode_consistency", "fit_s"]
agg = raw.groupby("generator", sort=False)[KEY].agg(["mean", "std"])
agg.to_csv(f"{RES_DIR}/06_seeds_summary.csv")

def pm(m, s, d=3):
    return "" if pd.isna(m) else (f"{m:.{d}f}" if pd.isna(s) else f"{m:.{d}f} ± {s:.{d}f}")
nice = pd.DataFrame({k: [pm(agg.loc[g, (k, "mean")], agg.loc[g, (k, "std")], 0 if k in ("LOS_max", "fit_s") else 3)
                         for g in agg.index] for k in KEY}, index=agg.index)
nice.to_csv(f"{RES_DIR}/06_seeds_summary_pretty.csv")

# scorecard: same targets as 04, scored inside each seed, then counted
def score_seed(d):
    d = d.set_index("generator")
    s = pd.DataFrame({
        "modes_ok":         d["mode_abs_err"] <= 0.06,
        "extremes_ok":      d["extreme_cov"] >= 0.08,
        "max_ok":           d["LOS_max"] >= 90,
        "no_copying":       d["DCR_ratio"] >= 1.0,
        "nonmode_best3":    d["nonmode_prev_err"].rank() <= 3,
        "within_mode_best3": d["within_mode_err"].rank() <= 3,
        "extreme_modes_ok": d["extreme_mode_err"] <= 0.03})
    return s.sum(axis=1)
targets = raw.groupby("seed").apply(score_seed).T if raw["seed"].nunique() > 1 else score_seed(raw).to_frame(SEEDS[0])
targets = targets.reindex([g for g in ORDER if g in targets.index])
targets.columns = [f"seed {c}" for c in targets.columns]
targets["mean (of 7)"] = targets.mean(axis=1).round(2)
targets.to_csv(f"{RES_DIR}/06_seeds_targets.csv")
print("Targets met per seed (out of 7):"); print(targets.to_string()); print()
nice.T

In [ ]:
# Cell 8 — plots: the key metrics with error bars (SD over seeds)
import matplotlib.pyplot as plt
show = [("mode_abs_err", "Mode mix error"), ("extreme_mode_err", "Mode mix error in the extreme tail"),
        ("within_mode_err", "Within-mode error (not gameable)"), ("DCR_ratio", "DCR ratio (>= 1 = no copying)")]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (k, title) in zip(axes, show):
    m, s = agg[(k, "mean")], agg[(k, "std")].fillna(0)
    ax.barh(m.index, m.values, xerr=s.values, capsize=3, color=["#999"] * 4 + ["#4C72B0", "#C44E52"][:len(m) - 4])
    ax.set_title(title); ax.invert_yaxis()
    if k == "DCR_ratio": ax.axvline(1, ls="--", c="k", lw=1)
fig.tight_layout(); fig.savefig(f"{RES_DIR}/06_seeds_plots.png", dpi=150); plt.show()
print("Saved: 06_seeds_raw.csv, 06_seeds_summary.csv, 06_seeds_summary_pretty.csv, 06_seeds_targets.csv, 06_seeds_plots.png")